In [1]:
import torch
print(torch.__version__)       # should now show 2.6.0
print(torch.version.cuda)      # should show 12.1 if GPU build
print(torch.cuda.is_available())  

2.7.1+cu118
11.8
True


In [17]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F
import pandas as pd
import os
from tqdm import tqdm  # for progress bars

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Load FinBERT tokenizer + model
model_name = "nickmuchi/sec-bert-finetuned-finance-classification"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)
model.eval()

# Loop over years and collect results
all_results = []

batch_size = 128
for year in range(2005, 2026):  # extend to 2026 later
    file_path = f"./data/text/text_us_{year}.pkl"
    if not os.path.exists(file_path):
        print(f"Skipping {year}, file not found")
        continue

    df_year = pd.read_pickle(file_path)  # load yearly dataframe
    print(f"\nProcessing {year}, total rows: {len(df_year)}")

    # Merge rf + mgmt text (drop if both are missing)
    combined_texts = []
    gvkeys = []
    for _, row in df_year.iterrows():
        rf_text = str(row["rf"]) if pd.notna(row["rf"]) else ""
        mgmt_text = str(row["mgmt"]) if pd.notna(row["mgmt"]) else ""
        merged = (rf_text + " " + mgmt_text).strip()
        if merged:  # only keep if not empty
            combined_texts.append(merged)
            gvkeys.append(row["gvkey"])

    if not combined_texts:
        print(f"No text to process for {year}, skipping.")
        continue

    all_probs = []
    with torch.no_grad():
        print(f"Processing {len(combined_texts)} documents in batches of {batch_size}...")
        for i in tqdm(range(0, len(combined_texts), batch_size), desc=f"Year {year} batches"):
            batch_texts = combined_texts[i:i+batch_size]
            inputs = tokenizer(batch_texts, return_tensors="pt",
                               truncation=True, max_length=512,
                               padding=True)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            outputs = model(**inputs)
            probs = F.softmax(outputs.logits, dim=1)  # [batch, num_labels]
            all_probs.extend(probs.cpu().tolist())

    # Make a dataframe for this year
    df_probs = pd.DataFrame(all_probs, columns=model.config.id2label.values())
    df_probs["gvkey"] = gvkeys
    df_probs["year"] = year

    all_results.append(df_probs)
    print(f"Finished processing year {year}, collected {len(df_probs)} rows.\n")

# Combine all years into one dataframe
df_all = pd.concat(all_results, ignore_index=True)
print("All years combined:", len(df_all), "rows")

# Optionally save
df_all.to_csv("finbert_sentiment_rf_mgmt_2005_2025.csv", index=False)
print("Saved CSV: finbert_sentiment_rf_mgmt_2005_2025.csv")


Using device: cuda


c:\Users\shoai\anaconda3\envs\MLCOURSE\lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(



Processing 2005, total rows: 16857
Processing 16825 documents in batches of 128...


Year 2005 batches: 100%|██████████| 132/132 [05:36<00:00,  2.55s/it]


Finished processing year 2005, collected 16825 rows.


Processing 2006, total rows: 16553
Processing 16521 documents in batches of 128...


...

Finished processing year 2023, collected 17793 rows.


Processing 2024, total rows: 20352
Processing 20312 documents in batches of 128...


Year 2024 batches: 100%|██████████| 159/159 [08:56<00:00,  3.37s/it]


Finished processing year 2024, collected 20312 rows.


Processing 2025, total rows: 11644
Processing 11623 documents in batches of 128...


Year 2025 batches: 100%|██████████| 91/91 [05:50<00:00,  3.85s/it]


Finished processing year 2025, collected 11623 rows.

All years combined: 357980 rows
Saved CSV: finbert_sentiment_rf_mgmt_2005_2025.csv


In [18]:
df_all

,bearish,neutral,bullish,gvkey,year
0,0.010647,0.932460,0.056893,6831.0,2005
1,0.045190,0.817574,0.137236,11872.0,2005
2,0.032643,0.918990,0.048367,24783.0,2005
3,0.027416,0.901566,0.071018,61721.0,2005
4,0.048750,0.634627,0.316623,146117.0,2005
...,...,...,...,...,...
357975,0.078705,0.810629,0.110666,38646.0,2025
357976,0.085659,0.827344,0.086997,NaN,2025
357977,0.037908,0.804489,0.157603,NaN,2025
357978,0.086194,0.813965,0.099841,NaN,2025
